# Seaborn 03 - Seaborn Exercises

> **MLCourse · Data Science Foundations · 04_seaborn**

Twelve challenges to make the seaborn toolkit stick - four **Easy**, five
**Medium**, three **Hard**. Each challenge gives you a goal, a dataset, a
checklist of required elements, and a starter cell. No answers here: work
each one yourself, then compare against `04_seaborn_solutions`.

Difficulty guide:

- **Easy** - one or two well-chosen calls plus labeling polish.
- **Medium** - multiple steps: transforms, masks, sampling, or multi-panel
  composition.
- **Hard** - custom `FacetGrid` functions, side-by-side comparisons, and a
  full mini-dashboard on one matplotlib figure.

Skim notebook 01/02 docs (or `help(sns.boxplot)`) freely - that is not
cheating, that is the job.

### What you'll learn

- Turning requirements ("compare these groups", "show uncertainty") into
  concrete seaborn calls without copy-pasting from tutorials.
- Combining figure-level convenience with axes-level composition.
- The discipline of checklists: ordering, labels, titles, honest error bars,
  colorblind-safe palettes.

### Setup


### Same Jupyter-safe magic pattern as notebooks 01-02.


In [ ]:
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    pass

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sns.set_theme()  # consistent baseline for every challenge


def load_dataset(name):
    """Load a built-in seaborn dataset with a friendly offline hint."""
    try:
        return sns.load_dataset(name)
    except Exception as exc:
        print(f"[offline hint] Could not load '{name}': {exc}")
        print("  Check your connection or fetch the CSV from")
        print("  https://github.com/mwaskom/seaborn-data and use pd.read_csv.")
        return None


tips = load_dataset("tips")
titanic = load_dataset("titanic")
_penguins_raw = load_dataset("penguins")
penguins = _penguins_raw.dropna() if _penguins_raw is not None else None
mpg = load_dataset("mpg")
flights = load_dataset("flights")

_diamonds_raw = load_dataset("diamonds")


def dia(n=5000):
    """Reproducible diamonds sample for the heavy challenges."""
    if _diamonds_raw is None:
        return None
    return _diamonds_raw.sample(n, random_state=42)


for label, df in [("tips", tips), ("titanic", titanic), ("penguins", penguins),
                  ("mpg", mpg), ("flights", flights)]:
    print(f"{label:>9}: {df.shape if df is not None else 'NOT LOADED'}")
print(f" dia(n=5000): ready={dia() is not None}")


--------------------------------------------------------------------------

### Challenge 1 - Mean spend by day, ordered and honest *(Easy)*

**Dataset:** `tips`

**Goal:** show mean `total_bill` per day as a bar chart with uncertainty,
bars ordered from highest to lowest mean.

**Checklist**

- [ ] `sns.barplot` with the default mean estimator.
- [ ] An explicit `order=` derived from groupby means (not alphabetical).
- [ ] A bootstrap CI error bar (default is fine) - and a caption/title saying
      what the error bars ARE.
- [ ] Axis labels + title; despined spines for polish.

### Starter - Challenge 1


In [ ]:
day_order = (
    tips.groupby("day")["total_bill"].mean()
    .sort_values(ascending=False)
    .index.tolist()
)
print("order:", day_order)

# TODO: barplot with order=day_order, then labels/title/despine.


--------------------------------------------------------------------------

### Challenge 2 - ECDF with a threshold read-off *(Easy)*

**Dataset:** `penguins`

**Goal:** an ECDF of `body_mass_g` per species answering "what fraction of
each species weighs under 4000 g?" at a glance.

**Checklist**

- [ ] `sns.ecdfplot` with `hue="species"`.
- [ ] A `rugplot` layered on the same axes (raw observations visible).
- [ ] A dashed vertical reference line at 4000 g.
- [ ] Print the exact per-species fractions below 4000 g to verify your eyes.

### Starter - Challenge 2


In [ ]:
ax = None  # create via sns.ecdfplot(...) assigned to ax, then layer onto it

# TODO: ecdfplot + rugplot + axvline(4000), then compute/print fractions.


--------------------------------------------------------------------------

### Challenge 3 - Corner pairplot by origin *(Easy)*

**Dataset:** `mpg`

**Goal:** a scatterplot matrix comparing fuel economy metrics across car
origins, half-size thanks to `corner=True`.

**Checklist**

- [ ] Variables: `mpg`, `horsepower`, `weight`, `acceleration`, `displacement`.
- [ ] `hue="origin"` with a colorblind-safe palette.
- [ ] `diag_kind="kde"` for smooth diagonals.
- [ ] `corner=True` and a suptitle on `pp.figure`.

### Starter - Challenge 3


In [ ]:
vars_5 = ["mpg", "horsepower", "weight", "acceleration", "displacement"]

# TODO: pp = sns.pairplot(...); remember pp.figure.suptitle(...).


--------------------------------------------------------------------------

### Challenge 4 - Colorblind redesign *(Easy)*

**Dataset:** `tips`

**Goal:** take any chart from this module (suggest: boxplot of total_bill by
day) and re-issue it in a colorblind-safe palette, previewing palettes first.

**Checklist**

- [ ] Preview at least two qualitative palettes with `sns.palplot`.
- [ ] Rebuild the chart passing `palette="colorblind"` (per-call).
- [ ] Briefly demo the global route too (`sns.set_palette`) and restore the
      default afterwards.
- [ ] One-sentence comment on WHY red/green defaults can fail readers.

### Starter - Challenge 4


In [ ]:
base_chart = sns.boxplot(data=tips, x="day", y="total_bill",
                         hue="day", legend=False)  # recolor me
plt.show()

# TODO: palplot previews -> redesigned chart (per-call palette)
#       -> global set_palette demo -> restore "deep".


--------------------------------------------------------------------------

### Challenge 5 - Faceted regression per origin *(Medium)*

**Dataset:** `mpg`

**Goal:** does horsepower hurt mpg equally everywhere? One regression panel
per origin, each with its own fit line.

**Checklist**

- [ ] Figure-level `lmplot` (NOT axes-level regplot) with `col="origin"`.
- [ ] Matching `hue="origin"` so fits and points share colors.
- [ ] Translucent scatter (`scatter_kws={"alpha": ...}`).
- [ ] Shared axis labels via `g.set_axis_labels`; suptitle stating the claim.
- [ ] Sanity comment: how many cars per facet (value_counts) before trusting.

### Starter - Challenge 5


In [ ]:
print(mpg["origin"].value_counts())

# TODO: g = sns.lmplot(...col="origin", hue="origin"...); annotate + title it.


--------------------------------------------------------------------------

### Challenge 6 - One variable, three lenses *(Medium)*

**Dataset:** `penguins`

**Goal:** compare body mass across species using box, violin, AND ECDF -
same data, three encodings, one row of panels.

**Checklist**
- [ ] One matplotlib `Figure` via `plt.subplots(1, 3)`; every seaborn call
      receives its target `ax=`.
- [ ] Panel 1: `boxplot` x=species; Panel 2: `violinplot` with
      `inner="quart"`; Panel 3: `ecdfplot` with `hue=species`.
- [ ] Shared comparison where meaningful (`sharey=True` helps panels 1-2).
- [ ] Per-panel titles naming the encoding; overall suptitle.

### Starter - Challenge 6


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# TODO: fill all three panels using axes[0], axes[1], axes[2].


--------------------------------------------------------------------------

### Challenge 7 - Masked correlation heatmap *(Medium)*

**Dataset:** `penguins`

**Goal:** publication-grade correlation matrix of the four morphometric
columns - lower triangle only, annotated, diverging-centered.

**Checklist**

- [ ] `corr = penguins[num_cols].corr(numeric_only=True)` on bill length /
      depth, flipper, mass.
- [ ] Boolean mask hiding the ABOVE-diagonal cells (`np.triu(..., k=1)`).
- [ ] `annot=True, fmt=".2f"`, `annot_kws` font size tweak.
- [ ] Diverging cmap (`vlag` or `RdBu_r`), `center=0`, `vmin=-1`, `vmax=1`,
      `square=True`.
- [ ] Print the strongest absolute pair programmatically (stack + sort).

### Starter - Challenge 7


In [ ]:
num_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
corr = penguins[num_cols].corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# TODO: sns.heatmap(...) then extract & print top pairs.


--------------------------------------------------------------------------

### Challenge 8 - Hexbin jointplot on big data *(Medium)*

**Dataset:** `diamonds` (sampled!)

**Goal:** relationship between carat and price when scatter would be solid
ink - hexagonal binning with marginal context.

**Checklist**

- [ ] Use `dia()` (5000 rows, random_state=42) - never plot 50k+ raw points
      casually.
- [ ] `sns.jointplot(kind="hex")` on carat vs price.
- [ ] `marginal_ticks=True` and a sensible `height`.
- [ ] Suptitle noting both variables are right-skewed (why hex beats
      scatter here).
- [ ] Bonus: try tuning hex resolution via `joint_kws={"gridsize": ...}`.

### Starter - Challenge 8


In [ ]:
d5k = dia()
print(d5k[["carat", "price"]].describe().round(1))

# TODO: j = sns.jointplot(data=d5k, kind="hex", ...); j.figure.suptitle(...).


--------------------------------------------------------------------------

### Challenge 9 - Pivot-table heatmap with a centered midpoint *(Medium)*

**Dataset:** `tips`

**Goal:** average tip by day × meal time as a heatmap whose colors center on
a meaningful value (the overall median tip), so "typical" reads neutral.

**Checklist**

- [ ] `pivot_table(index="day", columns="time", values="tip",
      aggfunc="mean")`, days ordered Thu→Sun.
- [ ] `center=float(np.median(tips["tip"]))` - say why in a comment.
- [ ] Diverging cmap, `annot=True, fmt=".2f"`, thin white separators.
- [ ] Title + labeled colorbar (`cbar_kws={"label": ...}`).

Note: tips has no month column, so we adapt the classic month×day pivot to
day×time; the flights dataset offers a true month×year variant if you want
extra practice afterwards.

### Starter - Challenge 9


In [ ]:
day_order_cal = ["Thur", "Fri", "Sat", "Sun"]
pivot = tips.pivot_table(index="day", columns="time", values="tip",
                         aggfunc="mean", observed=False).reindex(day_order_cal)
center_val = float(np.median(tips["tip"]))

# TODO: sns.heatmap(pivot, center=center_val, ...) with annotations & polish.


--------------------------------------------------------------------------

### Challenge 10 - FacetGrid with percentile bands *(Hard)*

**Dataset:** `tips`

**Goal:** tip vs total_bill faceted by day, where every facet carries its
own 25th/75th percentile lines for tip (and a median), drawn by YOUR custom
function through `map_dataframe`.

**Checklist**

- [ ] `FacetGrid(tips, col="day", col_wrap=2)` with scatter base layer.
- [ ] A custom function `(data, **kws)` drawing q25/q75 dashed lines +
      median solid line on `plt.gca()`, mapped via `g.map_dataframe(fn)`.
- [ ] `g.refline(y=...)` showing the GLOBAL median tip for contrast.
- [ ] Templated titles (`g.set_titles(col_template="{col_name}")`), legend
      or inline text explaining the line styles.
- [ ] No hardcoded percentile numbers - computed inside the function.

### Starter - Challenge 10


In [ ]:
def draw_tip_bands(data, **kws):
    """TODO: q25/median/q75 horizontal lines on the current axes."""
    ax = plt.gca()
    # TODO: quantile([0.25, 0.5, 0.75]) -> axhline trio (styles differ)


g = sns.FacetGrid(tips, col="day", col_wrap=2, height=3.0)
# TODO: g.map_dataframe(sns.scatterplot, ...) ; g.map_dataframe(draw_tip_bands)
# TODO: refline, titles, suptitle, plt.show()


--------------------------------------------------------------------------

### Challenge 11 - Box vs boxen showdown *(Hard)*

**Dataset:** `diamonds` (sampled)

**Goal:** argue visually for boxenplot on large n: same variable, two
encodings, shared scale, annotated sample sizes.

**Checklist**

- [ ] One figure, `plt.subplots(1, 2, sharey=True)`: left `boxplot`, right
      `boxenplot` of carat by cut (ordered worst→best cut).
- [ ] Identical y-limits on both panels (shared axes handle it - verify).
- [ ] n per cut printed AND annotated above one of the panels.
- [ ] A takeaway comment: what extra structure does boxen reveal (tail
      boxes), and when would you still prefer a plain box?

### Starter - Challenge 11


In [ ]:
d5k = dia()
cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
n_per_cut = d5k["cut"].value_counts().reindex(cut_order)
print(n_per_cut)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)

# TODO: boxplot on axes[0]; boxenplot on axes[1]; titles + n annotations.


--------------------------------------------------------------------------

### Challenge 12 - Mini-dashboard: four views, one figure *(Hard)*

**Dataset:** your pick across `tips` / `penguins`

**Goal:** a single 2×2 matplotlib figure hosting four DIFFERENT seaborn
axes-level plots, reading like a tipping-behavior dashboard.

**Checklist**

- [ ] `fig, axes = plt.subplots(2, 2, figsize=(12, 8))` - every seaborn call
      gets an `ax=` (no figure-level functions allowed here!).
- [ ] Panel A: distribution view (box or violin of tip by day).
- [ ] Panel B: the Challenge-9-style pivot heatmap (cbar off to save space).
- [ ] Panel C: interaction view - pointplot of tip by day, hue=smoker.
- [ ] Panel D: ECDF of penguin body mass by species (yes, mixing datasets on
      one dashboard is fine - label clearly).
- [ ] Per-panel titles + a `fig.suptitle`; `fig.tight_layout()` before show.

### Starter - Challenge 12


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# TODO: four axes-level calls into axes[0,0] .. axes[1,1].


### Wrap-up

When every checklist ticks green, open `04_seaborn_solutions` and diff your
approach against ours - different-but-valid is the norm in plotting; the
interesting conversation is *why* an encoding works. Then revisit any
challenge a week later: speed of recall is the real graduation test.